In [ ]:
# Load the autoreload extension
%load_ext autoreload

# Set autoreload mode
%autoreload 2

# Scraping Analysis & Discussion

In [1]:
from coralnet_scraper import CoralNetDownloader
import getpass
import boto3
import numpy as np
import pandas as pd
import tqdm

In [2]:
def check_s3_prefix_exists(bucket_name, s3_prefix, source_id, specific_file = "annotations.csv"):
    s3 = boto3.client("s3")
    prefix = f"{s3_prefix}/s{source_id}/{specific_file}"

    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix, MaxKeys=1)

    if "Contents" in response:
        # print(f"Prefix exists: {prefix}")
        return True
    else:
        # print(f"Prefix does not exist: {prefix}")
        return False

# Get sources with the image_list bug

In [3]:
df_counts = pd.read_csv("../nbs/EDA/dataframes/coralnet_source_counts.csv")
df_counts

,source_id,image_list_count,annotations_count,images_folder_count
0,23,750,750,750
1,57,1681,1681,1681
2,69,100,100,100
3,70,300,300,300
4,82,1,0,1
...,...,...,...,...
1956,8277,0,0,0
1957,8278,0,0,0
1958,8284,0,0,0
1959,8288,0,0,0


## Image List Issue Fixes (check all combinations between mismatches in the future)

In [4]:
df_counts_problematic = df_counts[df_counts["image_list_count"]!=df_counts["images_folder_count"]]
df_counts_problematic = df_counts_problematic[df_counts_problematic["image_list_count"]<df_counts_problematic["images_folder_count"]]

In [6]:
print(f"There are {df_counts_problematic.shape[0]} problematic sources, containing a total of {df_counts_problematic['images_folder_count'].sum()} images.")

There are 32 problematic sources, containing a total of 550017 images.


In [7]:
df_counts_problematic.sort_values(by="images_folder_count").head(10)

,source_id,image_list_count,annotations_count,images_folder_count
908,5027,40,40,41
217,2616,1904,4904,4904
216,2615,943,4943,4943
150,2112,89,6089,6089
318,3015,1081,7081,7081
299,2947,1143,7143,7142
459,3466,1202,7202,7202
503,3551,1202,7202,7202
510,3567,1419,7419,7419
251,2795,1779,7779,7779


# Check individual source

In [8]:
username = input("CoralNet username: ")
password = getpass.getpass("CoralNet password: ")

CoralNet username:  ViktorDo
CoralNet password:  ········


In [5]:
source_ids = df_counts_problematic["source_id"].values

bucket_name = "dev-datamermaid-sm-sources"
prefix = "coralnet-public-images"
s3 = boto3.client("s3")

In [24]:
downloader = CoralNetDownloader(username=username, password=password)
for source_id in tqdm.tqdm(["2947", "3466", "3551", "3567", "2795"], desc = "sources"):
    df_images, success = downloader.get_images(source_id=source_id)
    if success==True and len(df_images)>0: 
        s3_key_output = f"{prefix}/imagelist/s{source_id}_image_list.csv"
        s3.put_object(
            Bucket=bucket_name,
            Key=s3_key_output,
            Body=df_images.to_csv(index=False)
        )

sources:   0%|          | 0/5 [00:00<?, ?it/s]

Fetching images: 0page [00:00, ?page/s]

sources:  20%|██        | 1/5 [08:16<33:06, 496.62s/it]

Fetching images: 0page [00:00, ?page/s]

sources:  40%|████      | 2/5 [17:34<26:37, 532.49s/it]

...

sources:  60%|██████    | 3/5 [30:29<21:26, 643.41s/it]

Fetching images: 0page [00:00, ?page/s]

sources:  80%|████████  | 4/5 [39:48<10:09, 609.93s/it]

Fetching images: 0page [00:00, ?page/s]

sources: 100%|██████████| 5/5 [47:52<00:00, 574.50s/it]


In [7]:
df_images.head()

NameError: name 'df_images' is not defined

In [20]:
success

True

In [13]:
images_prefix = f"{prefix}/imagelist/"

paginator = s3.get_paginator("list_objects_v2")
file_count = 0

image_list = []
for page in paginator.paginate(Bucket=bucket_name, Prefix=images_prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != images_prefix and not key.endswith("/"):
            image_list.append(key)
            file_count += 1


In [14]:
len(image_list)

32

In [44]:
s3_key_image = f"{prefix}/s{source_id}/image_list.csv"
obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
image_list_df = pd.read_csv(obj_il["Body"])

In [45]:
s3_key_annotations = f"{prefix}/s{source_id}/annotations.csv"
obj_ann = s3.get_object(Bucket=bucket_name, Key=s3_key_annotations)
annotations_df = pd.read_csv(obj_ann["Body"])

/tmp/ipykernel_2801/1466276775.py:3: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  annotations_df = pd.read_csv(obj_ann["Body"])


In [11]:
images_prefix = f"{prefix}/s{source_id}/images/"

paginator = s3.get_paginator("list_objects_v2")
file_count = 0

image_list = []
for page in paginator.paginate(Bucket=bucket_name, Prefix=images_prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != images_prefix and not key.endswith("/"):
            image_list.append(key)
            file_count += 1

NameError: name 'source_id' is not defined

In [47]:
annotations_df.head(3)

,Name,Date,Season,Shelf-position,Reef,Exposure,Zone,Height (cm),Latitude,Longitude,...,Strobes,Framing gear used,White balance card,Comments,Row,Column,Label code,Label ID,Annotator,Date annotated
0,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,No,No,No,NaN,547,999,Pavement,2513,robot,2022-01-18 14:34:19+00:00
1,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,No,No,No,NaN,290,576,Pavement,2513,robot,2022-01-18 14:34:19+00:00
2,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,No,No,No,NaN,641,733,Shadow,222,robot,2022-01-18 14:34:19+00:00


In [48]:
image_list_df.head(3)

,Name,Image Page,Image URL
0,2019_Summ_Shark_Lee_Crest_T3IMG_0777.JPG - Con...,/image/2064680/view/,https://coralnet-production.s3.amazonaws.com/m...
1,2019_Summ_Shark_Lee_Crest_T3IMG_0778.JPG - Con...,/image/2064684/view/,https://coralnet-production.s3.us-west-2.amazo...
2,2019_Summ_Shark_Lee_Crest_T3IMG_0779.JPG - Con...,/image/2064688/view/,https://coralnet-production.s3.amazonaws.com/m...


In [38]:
df_images.head()

,Name,Image Page
0,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG - Un...,/image/2274993/view/
1,2019_Fall_Fahal_Lee_Crest_T1.IMG_1671.JPG - Un...,/image/2274996/view/
2,2019_Fall_Fahal_Lee_Crest_T1.IMG_1672.JPG - Un...,/image/2275000/view/
3,2019_Fall_Fahal_Lee_Crest_T1.IMG_1673.JPG - Un...,/image/2275003/view/
4,2019_Fall_Fahal_Lee_Crest_T1.IMG_1674.JPG - Un...,/image/2275006/view/


In [52]:
image_list_df["Status"] = None
confirmed_mask = image_list_df["Name"].apply(lambda x: "Confirmed" in x)
Unconfirmed_mask = image_list_df["Name"].apply(lambda x: "Unconfirmed" in x)
image_list_df.loc[confirmed_mask, "Status"] = "Confirmed"
image_list_df.loc[Unconfirmed_mask, "Status"] = "Unconfirmed"

df_images["Status"] = None
confirmed_mask = df_images["Name"].apply(lambda x: "Confirmed" in x)
Unconfirmed_mask = df_images["Name"].apply(lambda x: "Unconfirmed" in x)
df_images.loc[confirmed_mask, "Status"] = "Confirmed"
df_images.loc[Unconfirmed_mask, "Status"] = "Unconfirmed"

In [55]:
image_list_df["Status"].value_counts()

Status
Unconfirmed    1301
Confirmed       603
Name: count, dtype: int64

In [56]:
df_images["Status"].value_counts()

Status
Unconfirmed    2497
Confirmed      2407
Name: count, dtype: int64

In [49]:
# Extract image names from both dataframes
df_images_names = set(df_images['Name'].str.extract(r'([^-]+)')[0])
image_list_names = set(image_list_df['Name'].str.extract(r'([^-]+)')[0])

# Find intersection
intersection = df_images_names & image_list_names

print(f"Intersection count: {len(intersection)}")
print(f"Images in df_images only: {len(df_images_names - image_list_names)}")
print(f"Images in image_list_df only: {len(image_list_names - df_images_names)}")

Intersection count: 1904
Images in df_images only: 3000
Images in image_list_df only: 0


In [50]:
annotations_df["Name"].nunique(), len(image_list_df), len(image_list), len(df_images)

(4904, 1904, 4904, 4904)

In [69]:
annotations_df["Name"].apply(lambda x: "Confirmed" in x).sum(), annotations_df["Name"].apply(lambda x: "Unconfirmed" in x).sum()

(0, 0)

In [70]:
df_annotations = annotations_df.copy()
df_images_copy = df_images.copy()
df_images["Name"] = df_images["Name"].apply(
    lambda x: x.replace(" - Confirmed", "")
)
df_images["image_id"] = df_images["Image Page"].apply(
    lambda x: x.replace("/image/", "").replace("/view/", "")
)
df_annotations_copy = pd.merge(
    df_annotations,
    df_images,
    left_on="Name",
    right_on="Name",
    how="left",
    suffixes=("", "_y"),
)

In [73]:
df_images

,Name,Image Page,Status,image_id
0,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG - Un...,/image/2274993/view/,Unconfirmed,2274993
1,2019_Fall_Fahal_Lee_Crest_T1.IMG_1671.JPG - Un...,/image/2274996/view/,Unconfirmed,2274996
2,2019_Fall_Fahal_Lee_Crest_T1.IMG_1672.JPG - Un...,/image/2275000/view/,Unconfirmed,2275000
3,2019_Fall_Fahal_Lee_Crest_T1.IMG_1673.JPG - Un...,/image/2275003/view/,Unconfirmed,2275003
4,2019_Fall_Fahal_Lee_Crest_T1.IMG_1674.JPG - Un...,/image/2275006/view/,Unconfirmed,2275006
...,...,...,...,...
4899,2022_Wint_Shark_Lee_Crest_T3IMG_2486.JPG - Unc...,/image/2292638/view/,Unconfirmed,2292638
4900,2022_Wint_Shark_Lee_Crest_T3IMG_2487.JPG - Unc...,/image/2292639/view/,Unconfirmed,2292639
4901,2022_Wint_Shark_Lee_Crest_T3IMG_2488.JPG - Unc...,/image/2292640/view/,Unconfirmed,2292640
4902,2022_Wint_Shark_Lee_Crest_T3IMG_2490.JPG - Unc...,/image/2292643/view/,Unconfirmed,2292643


In [74]:
df_annotations_copy["Status"].value_counts()

Status
Confirmed    48140
Name: count, dtype: int64

In [71]:
df_annotations_copy

,Name,Date,Season,Shelf-position,Reef,Exposure,Zone,Height (cm),Latitude,Longitude,...,Comments,Row,Column,Label code,Label ID,Annotator,Date annotated,Image Page,Status,image_id
0,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,NaN,547,999,Pavement,2513,robot,2022-01-18 14:34:19+00:00,NaN,NaN,NaN
1,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,NaN,290,576,Pavement,2513,robot,2022-01-18 14:34:19+00:00,NaN,NaN,NaN
2,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,NaN,641,733,Shadow,222,robot,2022-01-18 14:34:19+00:00,NaN,NaN,NaN
3,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,NaN,352,1751,Pavement,2513,robot,2022-01-18 14:34:19+00:00,NaN,NaN,NaN
4,2019_Fall_Fahal_Lee_Crest_T1.IMG_1670.JPG,2021-10-31,Fall,Midshelf,Al Fahal,Sheltered,Crest,50,NaN,NaN,...,NaN,310,2642,Pavement,2513,robot,2022-01-18 14:34:19+00:00,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
98075,2022_Wint_Shark_Lee_Crest_T3IMG_2491.JPG,2022-01-19,Winter,Midshelf,Shark Reef,Sheltered,Crest,50,NaN,NaN,...,NaN,3076,717,CCA,101,robot,2022-01-27 08:23:51+00:00,NaN,NaN,NaN
98076,2022_Wint_Shark_Lee_Crest_T3IMG_2491.JPG,2022-01-19,Winter,Midshelf,Shark Reef,Sheltered,Crest,50,NaN,NaN,...,NaN,3172,786,Shadow,222,robot,2022-01-27 08:23:51+00:00,NaN,NaN,NaN
98077,2022_Wint_Shark_Lee_Crest_T3IMG_2491.JPG,2022-01-19,Winter,Midshelf,Shark Reef,Sheltered,Crest,50,NaN,NaN,...,NaN,3129,1458,Shadow,222,robot,2022-01-27 08:23:51+00:00,NaN,NaN,NaN
98078,2022_Wint_Shark_Lee_Crest_T3IMG_2491.JPG,2022-01-19,Winter,Midshelf,Shark Reef,Sheltered,Crest,50,NaN,NaN,...,NaN,3130,3473,DHC w/ Alg,635,robot,2023-06-02 11:43:35+00:00,NaN,NaN,NaN


# Confirmed Analysis

In [83]:
image_count_dict = {}

for source_id in tqdm.tqdm(df_counts["source_id"].values):
    image_list_flag = check_s3_prefix_exists(
        bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="image_list.csv"
    )
    if not image_list_flag:
        # print(f"Source {source_id} does not have an image_list.csv file. Skipping.")
        continue
    s3_key_image = f"{prefix}/s{source_id}/image_list.csv"
    obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
    image_list_df = pd.read_csv(obj_il["Body"])

    image_list_df["Status"] = "Other"
    confirmed_mask = image_list_df["Name"].apply(lambda x: "Confirmed" in x)
    Unconfirmed_mask = image_list_df["Name"].apply(lambda x: "Unconfirmed" in x or "Unclassified" in x)
    image_list_df.loc[confirmed_mask, "Status"] = "Confirmed"
    image_list_df.loc[Unconfirmed_mask, "Status"] = "Unconfirmed"
    if "Other" in image_list_df["Status"].unique():
        print(f"Source {source_id} has {image_list_df[image_list_df['Status']=='Other'].shape[0]} images with 'Other' status.")
        if image_list_df[image_list_df["Status"] == "Other"].shape[0] > 100:
            break
    for status in image_list_df["Status"].unique():
        count = image_list_df[image_list_df["Status"] == status].shape[0]
        if status not in image_count_dict:
            image_count_dict[status] = 0
        image_count_dict[status] += count


  0%|          | 1/1961 [00:00<05:13,  6.25it/s]

100%|██████████| 1961/1961 [04:19<00:00,  7.56it/s]


In [84]:
# image_list_df[image_list_df["Status"] == "Other"]["Name"].values

In [86]:
image_count_dict

{'Confirmed': 486155, 'Unconfirmed': 180677}

In [63]:
image_list_df

,Name,Image Page,Image URL
0,BF1_20151012_MA062_01.JPG - Confirmed,/image/2757199/view/,https://coralnet-production.s3.us-west-2.amazo...
1,BF1_20151012_MA062_03.JPG - Confirmed,/image/2757200/view/,https://coralnet-production.s3.us-west-2.amazo...
2,BF1_20151012_MA062_07.JPG - Confirmed,/image/2755058/view/,https://coralnet-production.s3.us-west-2.amazo...
3,BF1_20151012_MA062_09.JPG - Confirmed,/image/2757201/view/,https://coralnet-production.s3.us-west-2.amazo...
4,BF1_20151012_MA062_13.JPG - Confirmed,/image/2752853/view/,https://coralnet-production.s3.us-west-2.amazo...
...,...,...,...
8855,STM_22_20161111_T5_02.JPG - Confirmed,/image/2756978/view/,https://coralnet-production.s3.us-west-2.amazo...
8856,STM_22_20161111_T5_03.JPG - Confirmed,/image/2753370/view/,https://coralnet-production.s3.us-west-2.amazo...
8857,STM_22_20161111_T5_04.JPG - Confirmed,/image/2761964/view/,https://coralnet-production.s3.us-west-2.amazo...
8858,STM_22_20161111_T5_11.JPG - Confirmed,/image/2761965/view/,https://coralnet-production.s3.amazonaws.com/m...


# Check counts of new sources

In [20]:
prefix, bucket_name

('coralnet-public-images', 'dev-datamermaid-sm-sources')

In [21]:
images_prefix = f"{prefix}/imagelist/"

paginator = s3.get_paginator("list_objects_v2")
file_count = 0

image_list = []
for page in paginator.paginate(Bucket=bucket_name, Prefix=images_prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key != images_prefix and not key.endswith("/"):
            image_list.append(key)
            file_count += 1


In [29]:
image_list

['coralnet-public-images/imagelist/s1580_image_list.csv',
 'coralnet-public-images/imagelist/s2112_image_list.csv',
 'coralnet-public-images/imagelist/s2615_image_list.csv',
 'coralnet-public-images/imagelist/s2616_image_list.csv',
 'coralnet-public-images/imagelist/s2795_image_list.csv',
 'coralnet-public-images/imagelist/s2897_image_list.csv',
 'coralnet-public-images/imagelist/s2947_image_list.csv',
 'coralnet-public-images/imagelist/s2959_image_list.csv',
 'coralnet-public-images/imagelist/s3015_image_list.csv',
 'coralnet-public-images/imagelist/s3058_image_list.csv',
 'coralnet-public-images/imagelist/s3294_image_list.csv',
 'coralnet-public-images/imagelist/s3363_image_list.csv',
 'coralnet-public-images/imagelist/s3371_image_list.csv',
 'coralnet-public-images/imagelist/s3411_image_list.csv',
 'coralnet-public-images/imagelist/s3465_image_list.csv',
 'coralnet-public-images/imagelist/s3466_image_list.csv',
 'coralnet-public-images/imagelist/s3479_image_list.csv',
 'coralnet-pub

In [23]:
csv_dfs = {
    key: pd.read_csv(s3.get_object(Bucket=bucket_name, Key=key)["Body"])
    for key in image_list
}

In [25]:
image_list[-1]

'coralnet-public-images/imagelist/s8297_image_list.csv'

In [24]:
s3_key_output

'coralnet-public-images/imagelist/s372_image_list.csv'

In [30]:
df_counts_problematic = df_counts[df_counts["image_list_count"]!=df_counts["images_folder_count"]]
df_counts_problematic = df_counts_problematic[df_counts_problematic["image_list_count"]<df_counts_problematic["images_folder_count"]]
df_counts_problematic

,source_id,image_list_count,annotations_count,images_folder_count
12,295,1064,79064,79062
16,372,98,66098,66098
17,373,2485,29485,29485
24,554,1395,22395,22393
84,1580,1421,16421,16421
150,2112,89,6089,6089
216,2615,943,4943,4943
217,2616,1904,4904,4904
251,2795,1779,7779,7779
292,2897,2512,29512,29512


In [34]:
image_list_count_upd = []
for source_id in df_counts_problematic["source_id"]:
    try: 
        s3_key_output = f"{prefix}/imagelist/s{source_id}_image_list.csv"
        obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_output)
        image_list_df = pd.read_csv(obj_il["Body"])
        image_list_count_upd.append(image_list_df.shape[0])
    except Exception as e:
        print(f"Error occurred while processing source {source_id}: {e}")
        image_list_count_upd.append(np.nan)
        continue

Error occurred while processing source 295: An error occurred (NoSuchKey) when calling the GetObject operation: The specified key does not exist.


In [35]:
df_counts_problematic["image_list_count_upd"] = image_list_count_upd

In [36]:
df_counts_problematic

,source_id,image_list_count,annotations_count,images_folder_count,image_list_count_upd
12,295,1064,79064,79062,NaN
16,372,98,66098,66098,66098.0
17,373,2485,29485,29485,29485.0
24,554,1395,22395,22393,22395.0
84,1580,1421,16421,16421,16421.0
150,2112,89,6089,6089,6089.0
216,2615,943,4943,4943,4943.0
217,2616,1904,4904,4904,4904.0
251,2795,1779,7779,7779,7779.0
292,2897,2512,29512,29512,37454.0


# Confirmed Reanalysis

In [38]:
image_count_dict = {}

for _, row in tqdm.tqdm(df_counts.iterrows()):
    source_id = row["source_id"]
    problematic = row["image_list_count"]<row["images_folder_count"]
    if problematic and source_id!=295:
        s3_key_output = f"{prefix}/imagelist/s{source_id}_image_list.csv"
        obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_output)
        image_list_df = pd.read_csv(obj_il["Body"])
    else:
        image_list_flag = check_s3_prefix_exists(
            bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="image_list.csv"
        )
        if not image_list_flag:
            # print(f"Source {source_id} does not have an image_list.csv file. Skipping.")
            continue
        s3_key_image = f"{prefix}/s{source_id}/image_list.csv"
        obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_image)
        image_list_df = pd.read_csv(obj_il["Body"])

    image_list_df["Status"] = "Other"
    confirmed_mask = image_list_df["Name"].apply(lambda x: "Confirmed" in x)
    Unconfirmed_mask = image_list_df["Name"].apply(lambda x: "Unconfirmed" in x)
    Unclassified_mask = image_list_df["Name"].apply(lambda x: "Unclassified" in x)
    image_list_df.loc[confirmed_mask, "Status"] = "Confirmed"
    image_list_df.loc[Unconfirmed_mask, "Status"] = "Unconfirmed"
    image_list_df.loc[Unclassified_mask, "Status"] = "Unclassified"
    if "Other" in image_list_df["Status"].unique():
        print(f"Source {source_id} has {image_list_df[image_list_df['Status']=='Other'].shape[0]} images with 'Other' status.")
        if image_list_df[image_list_df["Status"] == "Other"].shape[0] > 100:
            break
    for status in image_list_df["Status"].unique():
        count = image_list_df[image_list_df["Status"] == status].shape[0]
        if status not in image_count_dict:
            image_count_dict[status] = 0
        image_count_dict[status] += count

1it [00:00,  4.10it/s]

1961it [04:20,  7.54it/s]


In [39]:
image_count_dict

{'Confirmed': 733661, 'Unconfirmed': 343470, 'Unclassified': 29470}

# Find eligible images

In [6]:
import time

In [12]:
start_time = time.time()
df_annotation_list = []
# for i, source_id in tqdm.tqdm(enumerate(source_ids)):
for row_i, row in tqdm.tqdm(df_counts.iterrows()):
    # if row_i==1000:
    #     break
    source_id = row["source_id"]
    problematic = row["image_list_count"]<row["images_folder_count"]

    annotations_flag = check_s3_prefix_exists(
        bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="annotations.csv"
    )
    if not annotations_flag:
        # print(f"Source {source_id} does not have an annotations.csv file. Skipping.")
        continue

    image_list_flag = check_s3_prefix_exists(
        bucket_name=bucket_name, s3_prefix=prefix, source_id=source_id, specific_file="image_list.csv"
    )
    if not image_list_flag:
        # print(f"Source {source_id} does not have an image_list.csv file. Skipping.")
        continue


    df_annotations = pd.read_csv(
        f"s3://{bucket_name}/{prefix}/s{source_id}/annotations.csv", low_memory=False
    )
    

    if problematic and source_id!=295:
        s3_key_output = f"{prefix}/imagelist/s{source_id}_image_list.csv"
        obj_il = s3.get_object(Bucket=bucket_name, Key=s3_key_output)
        df_images = pd.read_csv(obj_il["Body"])
    else:
        df_images = pd.read_csv(
            f"s3://{bucket_name}/{prefix}/s{source_id}/image_list.csv", low_memory=False  # Perhaps this is unnecessary and can just use tha annotations as in Mermaid
        )

    df_images["Name"] = df_images["Name"].apply(
        lambda x: x.replace(" - Confirmed", "")
    )
    df_images["image_id"] = df_images["Image Page"].apply(
        lambda x: x.replace("/image/", "").replace("/view/", "")
    )
    df_annotations = pd.merge(
        df_annotations,
        df_images,
        left_on="Name",
        right_on="Name",
        how="inner",
        suffixes=("", "_y"),
    )
    df_annotations["source_id"] = source_id
    df_annotation_list.append(df_annotations[["source_id", "image_id", "Row", "Column", "Label ID"]])

df_annotations = pd.concat(
    df_annotation_list, ignore_index=True
)

end_time = time.time()
print("Time taken (seconds):", end_time - start_time)

1961it [11:25,  2.86it/s]


Time taken (seconds): 688.4862225055695


In [14]:
df_annotations

,source_id,image_id,Row,Column,Label ID
0,23,12805,735,1008,112
1,23,12805,663,1682,106
2,23,12805,955,1737,106
3,23,12805,1034,1431,105
4,23,12805,851,2036,106
...,...,...,...,...,...
21058839,7525,5807407,756,655,84
21058840,7525,5807407,767,779,84
21058841,7525,5807407,824,1043,4777
21058842,7525,5807407,815,924,2100


In [15]:
df_annotations.isna().sum()

source_id    0
image_id     0
Row          0
Column       0
Label ID     0
dtype: int64

In [ ]:
# df_annotations.to_csv("df_annotations.csv", index=False)

In [8]:
df_annotations = pd.read_csv("df_annotations.csv")

In [9]:
df_annotations["image_id"].nunique(), df_annotations["source_id"].nunique()

(732278, 1241)

In [7]:
bucket_name = "dev-datamermaid-sm-sources"
prefix = "coralnet-public-images"
s3 = boto3.client("s3")

In [10]:
df_images = df_annotations.drop_duplicates(subset=["source_id", "image_id"]).reset_index()

In [11]:
df_images.shape

(732278, 6)

In [ ]:
from PIL import Image
import io
import tqdm 

valid_list = []
for i, row in tqdm.tqdm(df_images.iterrows()):
    source_id = row["source_id"]
    image_id = row["image_id"]
    key = f"{prefix}/s{source_id}/images/{image_id}.jpg"
    try:
        s3.head_object(Bucket=bucket_name, Key=key)
        obj = s3.get_object(Bucket=bucket_name, Key=key)

        with Image.open(io.BytesIO(obj["Body"].read())) as img:
            img.verify()

        valid_list.append(True)
    except Exception as e:
        valid_list.append(e)

261it [01:01,  4.51it/s]

In [25]:
np.unique(valid_list, return_counts=True)

(array([ True]), array([1001]))

In [ ]:
df_images["validation"] = valid_list

In [ ]:
df_images.to_csv("df_images.csv", index=False)